In [1]:
import pandas as pd

DATA_PATH = r"C:\Users\navya\OneDrive\Desktop\minor\combined_cleaned.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
df.head()

Dataset loaded successfully!
Shape: (16137168, 80)


,Dst Port,Protocol,Timestamp,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,0,0,14/02/2018 08:31:01,112641719,3,0,0.0,0.0,0.0,0.0,...,0,0.0,0.0,0.0,0.0,56320859.5,139.300036,56320958.0,56320761.0,Benign
1,0,0,14/02/2018 08:33:50,112641466,3,0,0.0,0.0,0.0,0.0,...,0,0.0,0.0,0.0,0.0,56320733.0,114.551299,56320814.0,56320652.0,Benign
2,0,0,14/02/2018 08:36:39,112638623,3,0,0.0,0.0,0.0,0.0,...,0,0.0,0.0,0.0,0.0,56319311.5,301.934596,56319525.0,56319098.0,Benign
3,22,6,14/02/2018 08:40:13,6453966,15,10,1239.0,2273.0,744.0,0.0,...,32,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,Benign
4,22,6,14/02/2018 08:40:23,8804066,14,11,1143.0,2209.0,744.0,0.0,...,32,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,Benign


In [2]:
from sklearn.model_selection import train_test_split

# Separate features and target
X = df.drop(columns=["Label", "Timestamp"])
y = df["Label"]

# Take a 10% stratified sample
X_sample, _, y_sample, _ = train_test_split(
    X,
    y,
    train_size=0.10,
    random_state=42,
    stratify=y
)

print("Sample shape:", X_sample.shape)
print("\nClass distribution:")
print(y_sample.value_counts())

Sample shape: (1613716, 78)

Class distribution:
Label
Benign                      1339023
DDOS attack-HOIC              68601
DDoS attacks-LOIC-HTTP        57619
DoS attacks-Hulk              46191
Bot                           28619
FTP-BruteForce                19335
SSH-Bruteforce                18759
Infilteration                 16064
DoS attacks-SlowHTTPTest      13989
DoS attacks-GoldenEye          4151
DoS attacks-Slowloris          1099
DDOS attack-LOIC-UDP            173
Brute Force -Web                 61
Brute Force -XSS                 23
SQL Injection                     9
Name: count, dtype: int64


In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_sample,
    y_sample,
    test_size=0.20,
    random_state=42,
    stratify=y_sample
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining labels:", y_train.shape)
print("Testing labels:", y_test.shape)

Training features: (1290972, 78)
Testing features: (322744, 78)

Training labels: (1290972,)
Testing labels: (322744,)


In [4]:
import numpy as np

# Calculate correlation matrix on training data
corr_matrix = X_train.corr().abs()

# Get upper triangle
upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

# Identify highly correlated features
to_drop = [
    column for column in upper.columns
    if any(upper[column] > 0.95)
]

# Keep selected features
selected_features = [
    column for column in X_train.columns
    if column not in to_drop
]

X_train = X_train[selected_features]
X_test = X_test[selected_features]

print("Original features:", 78)
print("Removed features:", len(to_drop))
print("Selected features:", len(selected_features))
print("\nSelected feature names:")
print(selected_features)

Original features: 78
Removed features: 25
Selected features: 53

Selected feature names:
['Dst Port', 'Protocol', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts', 'Fwd Pkt Len Max', 'Fwd Pkt Len Min', 'Fwd Pkt Len Mean', 'Bwd Pkt Len Max', 'Bwd Pkt Len Min', 'Bwd Pkt Len Mean', 'Flow Byts/s', 'Flow Pkts/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Fwd IAT Std', 'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Pkts/s', 'Bwd Pkts/s', 'Pkt Len Min', 'Pkt Len Max', 'Pkt Len Mean', 'Pkt Len Var', 'FIN Flag Cnt', 'RST Flag Cnt', 'PSH Flag Cnt', 'ACK Flag Cnt', 'URG Flag Cnt', 'Down/Up Ratio', 'Fwd Byts/b Avg', 'Fwd Pkts/b Avg', 'Fwd Blk Rate Avg', 'Bwd Byts/b Avg', 'Bwd Pkts/b Avg', 'Bwd Blk Rate Avg', 'Init Fwd Win Byts', 'Init Bwd Win Byts', 'Fwd Seg Size Min', 'Active Mean', 'Active Std', 'Active Max', 'Active Min', 'Idle Mean', 'Idle Std']


In [5]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer

# Replace infinite values
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)

# Fit imputer only on training data
imputer = SimpleImputer(strategy="median")

X_train = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=selected_features,
    index=X_train.index
)

X_test = pd.DataFrame(
    imputer.transform(X_test),
    columns=selected_features,
    index=X_test.index
)

print("Remaining missing values:", X_train.isna().sum().sum())
print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Remaining missing values: 0
Training shape: (1290972, 53)
Testing shape: (322744, 53)


In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training scaled:", X_train_scaled.shape)
print("Testing scaled:", X_test_scaled.shape)
print("Scaling completed!")

Training scaled: (1290972, 53)
Testing scaled: (322744, 53)
Scaling completed!


In [7]:
import os
import joblib
import json

MODEL_DIR = r"C:\Users\navya\OneDrive\Desktop\minor\AI-NIDS-SOC\models"
os.makedirs(MODEL_DIR, exist_ok=True)

joblib.dump(imputer, os.path.join(MODEL_DIR, "imputer.pkl"))
joblib.dump(scaler, os.path.join(MODEL_DIR, "scaler.pkl"))

with open(os.path.join(MODEL_DIR, "selected_features.json"), "w") as f:
    json.dump(selected_features, f, indent=4)

print("Preprocessing files saved successfully!")

Preprocessing files saved successfully!


In [8]:
from sklearn.preprocessing import LabelEncoder
import joblib
import os

label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

# Save label encoder
joblib.dump(
    label_encoder,
    os.path.join(MODEL_DIR, "label_encoder.pkl")
)

print("Number of classes:", len(label_encoder.classes_))
print("\nEncoded classes:")

for i, label in enumerate(label_encoder.classes_):
    print(i, ":", label)

Number of classes: 15

Encoded classes:
0 : Benign
1 : Bot
2 : Brute Force -Web
3 : Brute Force -XSS
4 : DDOS attack-HOIC
5 : DDOS attack-LOIC-UDP
6 : DDoS attacks-LOIC-HTTP
7 : DoS attacks-GoldenEye
8 : DoS attacks-Hulk
9 : DoS attacks-SlowHTTPTest
10 : DoS attacks-Slowloris
11 : FTP-BruteForce
12 : Infilteration
13 : SQL Injection
14 : SSH-Bruteforce


In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import time

rf_model = RandomForestClassifier(
    n_estimators=150,
    max_depth=25,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=2
)

start_time = time.time()

rf_model.fit(X_train_scaled, y_train_encoded)

training_time = time.time() - start_time

print(f"Training completed in {training_time/60:.2f} minutes")

Training completed in 6.21 minutes


In [10]:
from sklearn.metrics import accuracy_score, classification_report
import time

start = time.time()

y_pred = rf_model.predict(X_test_scaled)

prediction_time = time.time() - start

print("Accuracy:", round(accuracy_score(y_test_encoded, y_pred) * 100, 2), "%")
print(f"Prediction time: {prediction_time:.2f} seconds")

print("\nClassification Report:\n")

print(classification_report(
    y_test_encoded,
    y_pred,
    target_names=label_encoder.classes_,
    zero_division=0
))

Accuracy: 92.75 %
Prediction time: 3.84 seconds

Classification Report:

                          precision    recall  f1-score   support

                  Benign       0.99      0.93      0.96    267805
                     Bot       1.00      1.00      1.00      5724
        Brute Force -Web       0.80      0.67      0.73        12
        Brute Force -XSS       1.00      1.00      1.00         4
        DDOS attack-HOIC       1.00      1.00      1.00     13720
    DDOS attack-LOIC-UDP       0.78      0.91      0.84        35
  DDoS attacks-LOIC-HTTP       1.00      1.00      1.00     11524
   DoS attacks-GoldenEye       1.00      1.00      1.00       830
        DoS attacks-Hulk       1.00      1.00      1.00      9238
DoS attacks-SlowHTTPTest       0.55      0.96      0.70      2798
   DoS attacks-Slowloris       0.99      1.00      1.00       220
          FTP-BruteForce       0.93      0.43      0.59      3867
           Infilteration       0.06      0.40      0.11      3213
  

In [11]:
import joblib
import os

rf_path = os.path.join(MODEL_DIR, "random_forest.pkl")

joblib.dump(rf_model, rf_path)

print("Random Forest saved successfully!")
print("Location:", rf_path)

Random Forest saved successfully!
Location: C:\Users\navya\OneDrive\Desktop\minor\AI-NIDS-SOC\models\random_forest.pkl


In [14]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))

TensorFlow version: 2.22.0-rc1
GPU available: []


In [13]:
%pip install tensorflow

   ---------------------------------------- 0.0/382.7 MB ? eta -:--:--
   ---------------------------------------- 0.5/382.7 MB 4.6 MB/s eta 0:01:24
   ---------------------------------------- 1.8/382.7 MB 5.4 MB/s eta 0:01:11
   ---------------------------------------- 2.9/382.7 MB 5.3 MB/s eta 0:01:13
   ---------------------------------------- 4.2/382.7 MB 5.7 MB/s eta 0:01:07
    --------------------------------------- 5.8/382.7 MB 6.2 MB/s eta 0:01:01
    --------------------------------------- 7.1/382.7 MB 6.2 MB/s eta 0:01:01
    --------------------------------------- 7.9/382.7 MB 5.7 MB/s eta 0:01:06
    --------------------------------------- 8.7/382.7 MB 5.5 MB/s eta 0:01:09
    --------------------------------------- 8.9/382.7 MB 5.4 MB/s eta 0:01:10
   - -------------------------------------- 10.0/382.7 MB 5.1 MB/s eta 0:01:13
   - -------------------------------------- 10.5/382.7 MB 5.0 MB/s eta 0:01:15
   - -------------------------------------- 11.0/382.7 MB 4.6 MB/s et


[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
import numpy as np

# Select only benign training records
benign_indices = np.where(y_train_encoded == 0)[0]

# Randomly select up to 100,000 benign samples
rng = np.random.default_rng(42)

if len(benign_indices) > 100000:
    benign_indices = rng.choice(benign_indices, 100000, replace=False)

X_benign = X_train_scaled[benign_indices].astype(np.float32)

# Split benign data into training and validation
split = int(0.8 * len(X_benign))

X_ae_train = X_benign[:split]
X_ae_val = X_benign[split:]

print("Autoencoder training:", X_ae_train.shape)
print("Autoencoder validation:", X_ae_val.shape)

Autoencoder training: (80000, 53)
Autoencoder validation: (20000, 53)


In [16]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

autoencoder = Sequential([
    Input(shape=(53,)),

    Dense(32, activation="relu"),
    Dense(16, activation="relu"),
    Dense(8, activation="relu"),   # Bottleneck

    Dense(16, activation="relu"),
    Dense(32, activation="relu"),
    Dense(53, activation="linear")  # Reconstruction
])

autoencoder.compile(
    optimizer="adam",
    loss="mse"
)

autoencoder.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │         1,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 16)             │           144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │           544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 53)             │         1,749 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,829 (18.86 KB)

 Trainable params: 4,829 (18.86 KB)

 Non-trainable params: 0 (0.00 B)

In [17]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history = autoencoder.fit(
    X_ae_train,
    X_ae_train,
    validation_data=(X_ae_val, X_ae_val),
    epochs=20,
    batch_size=256,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.4494 - val_loss: 0.3033
Epoch 2/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.1966 - val_loss: 0.1953
Epoch 3/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.1418 - val_loss: 0.1617
Epoch 4/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.1186 - val_loss: 0.1400
Epoch 5/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.1033 - val_loss: 0.1242
Epoch 6/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0942 - val_loss: 0.1134
Epoch 7/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0861 - val_loss: 0.1011
Epoch 8/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0788 - val_loss: 0.0904
Epoch 9/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0725 - val_loss: 0.0820
Epoch 10/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0679 - val_loss: 0.0751
Epoch 11/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0636 - val_loss: 0.0698
Epoch 12/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step

In [18]:
# Calculate reconstruction error on benign validation data

X_ae_val_pred = autoencoder.predict(X_ae_val, verbose=0)

reconstruction_errors = np.mean(
    np.square(X_ae_val - X_ae_val_pred),
    axis=1
)

print("Mean reconstruction error:", np.mean(reconstruction_errors))
print("Maximum reconstruction error:", np.max(reconstruction_errors))
print("99th percentile:", np.percentile(reconstruction_errors, 99))

Mean reconstruction error: 0.04522798
Maximum reconstruction error: 22.418257
99th percentile: 0.63162696


In [19]:
import joblib
import os

threshold = float(np.percentile(reconstruction_errors, 99))

autoencoder.save(
    os.path.join(MODEL_DIR, "autoencoder.keras")
)

joblib.dump(
    threshold,
    os.path.join(MODEL_DIR, "autoencoder_threshold.pkl")
)

print("Autoencoder saved successfully!")
print("Threshold saved:", threshold)

Autoencoder saved successfully!
Threshold saved: 0.6316269636154175


In [20]:
# Reconstruct test data
X_test_pred = autoencoder.predict(
    X_test_scaled.astype(np.float32),
    batch_size=256,
    verbose=1
)

# Calculate reconstruction errors
test_errors = np.mean(
    np.square(X_test_scaled.astype(np.float32) - X_test_pred),
    axis=1
)

# Predict anomalies
y_ae_pred = (test_errors > threshold).astype(int)

# Actual labels: 0 = Benign, 1 = Attack
y_ae_true = (y_test_encoded != 0).astype(int)

print("Evaluation completed!")
print("Total test samples:", len(y_ae_true))
print("Predicted anomalies:", np.sum(y_ae_pred))

1261/1261 ━━━━━━━━━━━━━━━━━━━━ 1s 603us/step
Evaluation completed!
Total test samples: 322744
Predicted anomalies: 4266


In [21]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

print("AUTOENCODER EVALUATION")
print("----------------------")

print("Accuracy:", accuracy_score(y_ae_true, y_ae_pred))
print("Precision:", precision_score(y_ae_true, y_ae_pred, zero_division=0))
print("Recall:", recall_score(y_ae_true, y_ae_pred, zero_division=0))
print("F1 Score:", f1_score(y_ae_true, y_ae_pred, zero_division=0))
print("ROC-AUC:", roc_auc_score(y_ae_true, test_errors))

print("\nConfusion Matrix:")
print(confusion_matrix(y_ae_true, y_ae_pred))

print("\nClassification Report:")
print(classification_report(
    y_ae_true,
    y_ae_pred,
    target_names=["Benign", "Attack"],
    zero_division=0
))

AUTOENCODER EVALUATION
----------------------
Accuracy: 0.8270672731328855
Precision: 0.39756211908110645
Recall: 0.030870601940333826
F1 Score: 0.057292458407229116
ROC-AUC: 0.8314631141204097

Confusion Matrix:
[[265235   2570]
 [ 53243   1696]]

Classification Report:
              precision    recall  f1-score   support

      Benign       0.83      0.99      0.90    267805
      Attack       0.40      0.03      0.06     54939

    accuracy                           0.83    322744
   macro avg       0.62      0.51      0.48    322744
weighted avg       0.76      0.83      0.76    322744



In [22]:
from sklearn.metrics import precision_score, recall_score, f1_score

percentiles = [95, 97, 98, 99]

print("THRESHOLD COMPARISON")
print("-" * 65)

for p in percentiles:
    temp_threshold = np.percentile(reconstruction_errors, p)

    temp_pred = (test_errors > temp_threshold).astype(int)

    precision = precision_score(y_ae_true, temp_pred, zero_division=0)
    recall = recall_score(y_ae_true, temp_pred, zero_division=0)
    f1 = f1_score(y_ae_true, temp_pred, zero_division=0)

    print(
        f"Percentile: {p}% | "
        f"Threshold: {temp_threshold:.4f} | "
        f"Precision: {precision:.4f} | "
        f"Recall: {recall:.4f} | "
        f"F1: {f1:.4f}"
    )

THRESHOLD COMPARISON
-----------------------------------------------------------------
Percentile: 95% | Threshold: 0.1424 | Precision: 0.4855 | Recall: 0.2307 | F1: 0.3128
Percentile: 97% | Threshold: 0.2289 | Precision: 0.4663 | Recall: 0.1253 | F1: 0.1975
Percentile: 98% | Threshold: 0.3369 | Precision: 0.5498 | Recall: 0.1183 | F1: 0.1947
Percentile: 99% | Threshold: 0.6316 | Precision: 0.3976 | Recall: 0.0309 | F1: 0.0573


In [23]:
# Set candidate threshold at 95th percentile
threshold = float(np.percentile(reconstruction_errors, 95))

# Save updated threshold
joblib.dump(
    threshold,
    os.path.join(MODEL_DIR, "autoencoder_threshold.pkl")
)

print("Updated threshold:", threshold)
print("Threshold file updated successfully!")

Updated threshold: 0.14241467416286469
Threshold file updated successfully!


In [25]:
import sys
from pathlib import Path

PROJECT_DIR = Path(
    r"C:\Users\navya\OneDrive\Desktop\minor\AI-NIDS-SOC"
)

print("Project folder exists:", PROJECT_DIR.exists())
print("Pipeline exists:", (PROJECT_DIR / "src" / "pipeline.py").exists())

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from src.pipeline import detect_flows

print("Pipeline imported successfully!")

Project folder exists: True
Pipeline exists: True
AI-NIDS-SOC detection pipeline loaded successfully!
Pipeline imported successfully!


In [26]:
# Select 10 actual test samples
test_samples = X_test.iloc[:10].copy()

# Pass them through Random Forest + Autoencoder
results = detect_flows(test_samples)

# Display results
print("\nINTEGRATED PIPELINE RESULTS")
print("=" * 70)

print(results.to_string(index=False))

c:\Users\navya\OneDrive\Desktop\minor\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(



INTEGRATED PIPELINE RESULTS
         rf_prediction  rf_confidence  anomaly_score                              verdict
                Benign       0.930206       0.033157                                 Safe
                Benign       1.000000       0.025079                                 Safe
DDoS attacks-LOIC-HTTP       0.992447            NaN Known attack: DDoS attacks-LOIC-HTTP
                Benign       0.967752       0.002944                                 Safe
                Benign       0.787913       0.005401                                 Safe
                Benign       0.886136       0.008554                                 Safe
                Benign       0.526577       0.005213                                 Safe
                Benign       0.910618       0.042983                                 Safe
                Benign       0.971411       0.091004                                 Safe
                   Bot       0.975137            NaN                   

In [27]:
from src.risk import add_risk_scores

# Apply risk scoring to the previous pipeline results
final_results = add_risk_scores(results)

# Display the integrated results
print(final_results.to_string(index=False))

         rf_prediction  rf_confidence  anomaly_score                              verdict  risk_score severity
                Benign       0.930206       0.033157                                 Safe       23.26      Low
                Benign       1.000000       0.025079                                 Safe       25.00      Low
DDoS attacks-LOIC-HTTP       0.992447            NaN Known attack: DDoS attacks-LOIC-HTTP       99.81 Critical
                Benign       0.967752       0.002944                                 Safe       24.19      Low
                Benign       0.787913       0.005401                                 Safe       19.70      Low
                Benign       0.886136       0.008554                                 Safe       22.15      Low
                Benign       0.526577       0.005213                                 Safe       13.16      Low
                Benign       0.910618       0.042983                                 Safe       22.77      Low
 

In [28]:
from src.actions import add_recommended_actions

# Add SOC recommendations
soc_results = add_recommended_actions(final_results)

# Display complete results
print(soc_results.to_string(index=False))

         rf_prediction  rf_confidence  anomaly_score                              verdict  risk_score severity                     recommended_action
                Benign       0.930206       0.033157                                 Safe       23.26      Low                    Continue monitoring
                Benign       1.000000       0.025079                                 Safe       25.00      Low                    Continue monitoring
DDoS attacks-LOIC-HTTP       0.992447            NaN Known attack: DDoS attacks-LOIC-HTTP       99.81 Critical Prioritize immediate SOC investigation
                Benign       0.967752       0.002944                                 Safe       24.19      Low                    Continue monitoring
                Benign       0.787913       0.005401                                 Safe       19.70      Low                    Continue monitoring
                Benign       0.886136       0.008554                                 Safe       22.1

In [29]:
from src.reporting import save_detection_results

# Save the SOC results
report_path = save_detection_results(soc_results)

print("Report location:", report_path)

Detection report saved successfully: C:\Users\navya\OneDrive\Desktop\minor\AI-NIDS-SOC\reports\detection_results.csv
Report location: C:\Users\navya\OneDrive\Desktop\minor\AI-NIDS-SOC\reports\detection_results.csv


In [30]:
from src.pipeline import detect_flows
from src.risk import add_risk_scores
from src.actions import add_recommended_actions
from src.reporting import save_detection_results

# Process test data in batches
batch_size = 10000
all_results = []

for start in range(0, len(X_test), batch_size):
    end = min(start + batch_size, len(X_test))

    print(f"Processing rows {start} to {end}...")

    batch = X_test.iloc[start:end].copy()

    predictions = detect_flows(batch)
    predictions = add_risk_scores(predictions)
    predictions = add_recommended_actions(predictions)

    all_results.append(predictions)

# Combine all batch results
full_results = pd.concat(all_results, ignore_index=True)

print("\nTotal flows processed:", len(full_results))
print(full_results["severity"].value_counts())

Processing rows 0 to 10000...


c:\Users\navya\OneDrive\Desktop\minor\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


Processing rows 10000 to 20000...


c:\Users\navya\OneDrive\Desktop\minor\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


Processing rows 20000 to 30000...


c:\Users\navya\OneDrive\Desktop\minor\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


... (57 outputs left out)

c:\Users\navya\OneDrive\Desktop\minor\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


Processing rows 320000 to 322744...


c:\Users\navya\OneDrive\Desktop\minor\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(



Total flows processed: 322744
severity
Low         237866
Critical     72150
High         12728
Name: count, dtype: int64


In [31]:
save_detection_results(
    full_results,
    filename="full_detection_results.csv"
)

Detection report saved successfully: C:\Users\navya\OneDrive\Desktop\minor\AI-NIDS-SOC\reports\full_detection_results.csv


WindowsPath('C:/Users/navya/OneDrive/Desktop/minor/AI-NIDS-SOC/reports/full_detection_results.csv')